# Phase 4: Interactive Churn Risk Dashboard

## Building a Streamlit Dashboard for Customer Success Teams

**What this phase adds to the project:**

| Component | Technology | Purpose |
|---|---|---|
| **Dashboard** | Streamlit | Interactive web app for non-technical users |
| **Visualisations** | Plotly | Interactive charts (hover, zoom, filter) |
| **KPI Monitoring** | Metrics cards | At-a-glance business health |
| **Customer Drilldown** | Feature explorer | Individual risk profile analysis |
| **Data Export** | CSV download | Actionable customer lists |

**Why this matters for hiring managers:**
Building a model is one thing — making it *usable* for business teams is another. This dashboard turns raw predictions into actionable insights that a customer success manager can use without touching code. This demonstrates **product thinking** and **stakeholder communication** — skills that separate strong candidates from average ones.

---

## 0. Environment Setup (Google Colab)

In [ ]:
# ── Google Colab / Drive Setup ──
import sys
import os

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project path — UPDATE THIS if your folder structure differs
PROJECT_PATH = "/content/drive/MyDrive/Github/Projects/saas-churn-prediction"

# Change to project directory
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Project files: {os.listdir('.')[:10]}")

In [ ]:
# Install dependencies
!pip install streamlit plotly pandas numpy joblib -q

# Verify required data files exist (from Phase 2)
from pathlib import Path

data_dir = Path("data/processed")
models_dir = Path("models")

required_files = {
    "scored_customers.csv": data_dir / "scored_customers.csv",
    "features.csv": data_dir / "features.csv",
    "model_metadata.joblib": models_dir / "model_metadata.joblib",
}

print("Checking required files:")
for name, path in required_files.items():
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"  {status} {name} → {path}")

print("\n✓ Setup complete")

---
## 1. Understanding the Dashboard Architecture

### Why Streamlit?

Streamlit is the industry standard for data science dashboards because:
- **Python-native** — no JavaScript, HTML, or CSS required
- **Rapid prototyping** — build an interactive app in hours, not weeks
- **Data-first** — designed around DataFrames and plots
- **Free deployment** — Streamlit Community Cloud hosts apps for free

### Dashboard Structure

```
dashboard/
├── app.py              ← Main Streamlit application
└── requirements.txt    ← Dashboard-specific dependencies
```

### What the Dashboard Shows

| Section | Purpose | Business Value |
|---|---|---|
| **KPI Cards** | Total customers, churn rate, MRR at risk | Executive summary at a glance |
| **Risk Distribution** | Bar chart of risk levels | Where to focus retention efforts |
| **Revenue at Risk** | Revenue broken down by risk level | Prioritise by financial impact |
| **Probability Distribution** | Histogram with threshold line | Understand model confidence |
| **Revenue vs Probability** | Scatter plot | Find high-value at-risk customers |
| **Model Performance** | Metrics, threshold, revenue impact | Transparency and trust |
| **Customer Drilldown** | Individual customer analysis | Deep-dive for account managers |
| **High-Risk Table** | Sortable, downloadable customer list | Actionable retention list |

### How Data Flows

```
Phase 2 (Training)
    │
    ├── scored_customers.csv  ──→  Dashboard reads this
    ├── features.csv          ──→  For customer drilldown
    └── model_metadata.joblib ──→  For model info display
```

---
## 2. Exploring the Dashboard Data

Before launching the dashboard, let's explore the data it will display.

In [ ]:
import pandas as pd
import numpy as np
import joblib

# Load the same data the dashboard uses
scored_df = pd.read_csv("data/processed/scored_customers.csv")
features_df = pd.read_csv("data/processed/features.csv")
metadata = joblib.load("models/model_metadata.joblib")

print("═" * 60)
print("  SCORED CUSTOMERS DATA")
print("═" * 60)
print(f"  Total customers: {len(scored_df):,}")
print(f"  Columns: {list(scored_df.columns)}")
print(f"\n  Risk Distribution:")
print(scored_df["risk_level"].value_counts().to_string())
print(f"\n  Churn Prediction Distribution:")
print(scored_df["churn_prediction"].value_counts().to_string())
print(f"\n  Revenue Stats:")
print(f"    Mean monthly revenue:  £{scored_df['monthly_revenue'].mean():.2f}")
print(f"    Total MRR:             £{scored_df['monthly_revenue'].sum():,.2f}")
churners = scored_df[scored_df["churn_prediction"] == 1]
print(f"    MRR at risk:           £{churners['monthly_revenue'].sum():,.2f}")

In [ ]:
print("═" * 60)
print("  MODEL METADATA")
print("═" * 60)
print(f"  Best model:        {metadata.get('best_model', 'N/A')}")
print(f"  Optimal threshold: {metadata.get('optimal_threshold', 'N/A'):.4f}")

metrics = metadata.get("metrics", {})
print(f"\n  Metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"    {k:20s}: {v:.4f}")

rev = metadata.get("revenue_impact", {})
print(f"\n  Revenue Impact:")
for k, v in rev.items():
    print(f"    {k:30s}: £{v:,.2f}")

---
## 3. Dashboard Visualisations (Inline Preview)

Since Streamlit can't run inside Colab, we'll recreate the key dashboard charts here using the same Plotly code. This shows exactly what the dashboard displays.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# ── KPI Summary ──
total_customers = len(scored_df)
predicted_churners = scored_df[scored_df["churn_prediction"] == 1]
churn_rate = len(predicted_churners) / total_customers * 100
mrr_at_risk = predicted_churners["monthly_revenue"].sum()
critical_count = len(scored_df[scored_df["risk_level"] == "critical"])

print("═" * 60)
print("  DASHBOARD KPI CARDS")
print("═" * 60)
print(f"  Total Customers:     {total_customers:,}")
print(f"  Predicted Churners:  {len(predicted_churners):,}")
print(f"  Churn Rate:          {churn_rate:.1f}%")
print(f"  MRR at Risk:         £{mrr_at_risk:,.0f}")
print(f"  Critical Risk:       {critical_count:,}")

In [ ]:
# ── Risk Level Distribution ──
colour_map = {
    "low": "#2ecc71",
    "medium": "#f39c12",
    "high": "#e67e22",
    "critical": "#e74c3c",
}

risk_counts = (
    scored_df["risk_level"]
    .value_counts()
    .reindex(["low", "medium", "high", "critical"], fill_value=0)
    .reset_index()
)
risk_counts.columns = ["Risk Level", "Count"]

fig = px.bar(
    risk_counts,
    x="Risk Level",
    y="Count",
    color="Risk Level",
    color_discrete_map=colour_map,
    text="Count",
    title="Customer Risk Level Distribution",
)
fig.update_layout(showlegend=False, height=400)
fig.update_traces(textposition="outside")
fig.show()

In [ ]:
# ── Revenue at Risk by Risk Level ──
rev_by_risk = (
    scored_df[scored_df["churn_prediction"] == 1]
    .groupby("risk_level", observed=False)["monthly_revenue"]
    .sum()
    .reindex(["low", "medium", "high", "critical"], fill_value=0)
    .reset_index()
)
rev_by_risk.columns = ["Risk Level", "Monthly Revenue at Risk (£)"]

fig = px.bar(
    rev_by_risk,
    x="Risk Level",
    y="Monthly Revenue at Risk (£)",
    color="Risk Level",
    color_discrete_map=colour_map,
    text_auto=",.0f",
    title="Monthly Revenue at Risk by Risk Level",
)
fig.update_layout(showlegend=False, height=400)
fig.update_traces(textposition="outside")
fig.show()

In [ ]:
# ── Churn Probability Distribution with Threshold ──
threshold = metadata.get("optimal_threshold", 0.5)

fig = px.histogram(
    scored_df,
    x="churn_probability",
    nbins=30,
    color_discrete_sequence=["#3498db"],
    title="Churn Probability Distribution",
    labels={"churn_probability": "Churn Probability"},
)
fig.add_vline(
    x=threshold,
    line_dash="dash",
    line_color="#e74c3c",
    annotation_text=f"Threshold: {threshold:.2f}",
    annotation_position="top right",
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# ── Revenue vs Churn Probability Scatter ──
fig = px.scatter(
    scored_df,
    x="churn_probability",
    y="monthly_revenue",
    color="risk_level",
    color_discrete_map=colour_map,
    opacity=0.6,
    title="Monthly Revenue vs Churn Probability",
    labels={
        "churn_probability": "Churn Probability",
        "monthly_revenue": "Monthly Revenue (£)",
        "risk_level": "Risk Level",
    },
)
fig.update_layout(height=450)
fig.show()

---
## 4. Customer Drilldown Demo

This shows how the dashboard's drilldown feature works — selecting a customer and seeing their full risk profile.

In [ ]:
# ── Pick the highest-risk customer ──
top_risk = scored_df.sort_values("churn_probability", ascending=False).iloc[0]
cust_id = top_risk["customer_id"]

# Get their full feature profile
cust_features = features_df[features_df["customer_id"] == cust_id].iloc[0]

print("═" * 60)
print(f"  CUSTOMER DRILLDOWN: {cust_id}")
print("═" * 60)
print(f"  Churn Probability:   {top_risk['churn_probability']:.1%}")
print(f"  Risk Level:          {top_risk['risk_level'].upper()}")
print(f"  Monthly Revenue:     £{top_risk['monthly_revenue']:.2f}")
print(f"  Actual Churn:        {'Yes' if top_risk.get('actual_churn', 0) == 1 else 'No'}")

print(f"\n  ── Profile ──")
contract_map = {0: "Month-to-month", 1: "One year", 2: "Two year"}
print(f"  Tenure:              {int(cust_features.get('tenure_months', 0))} months")
print(f"  Contract:            {contract_map.get(int(cust_features.get('contract_type', 0)), 'Unknown')}")
print(f"  Total Charges:       £{cust_features.get('total_charges', 0):,.2f}")
print(f"  Login Frequency:     {cust_features.get('login_frequency', 0):.1f}")
print(f"  Feature Adoption:    {cust_features.get('feature_adoption_rate', 0):.0%}")
print(f"  Engagement Score:    {cust_features.get('engagement_score', 0):.1f}")
print(f"  Days Since Login:    {int(cust_features.get('days_since_last_login', 0))}")
print(f"  Support Tickets:     {int(cust_features.get('support_tickets_total', 0))}")

In [ ]:
# ── Feature importance for this customer ──
skip_cols = ["customer_id", "churn"]
feature_cols = [c for c in features_df.columns if c not in skip_cols]

cust_vals = cust_features[feature_cols].astype(float)
top_10 = cust_vals.abs().sort_values(ascending=True).tail(10)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top_10.values,
    y=top_10.index,
    orientation="h",
    marker_color=["#e74c3c" if cust_vals[f] > 0 else "#2ecc71" for f in top_10.index],
))
fig.update_layout(
    title=f"Top Features — Customer {cust_id}",
    height=400,
    xaxis_title="Feature Value (absolute)",
    margin=dict(l=10, r=10, t=40, b=10),
)
fig.show()

---
## 5. High-Risk Customer Table

This table is what a customer success manager would export and use to prioritise their outreach.

In [ ]:
# ── Show high-risk customers (what the dashboard's table displays) ──
high_risk = scored_df[scored_df["risk_level"].isin(["critical", "high"])].sort_values(
    "churn_probability", ascending=False
)

display_cols = ["customer_id", "churn_probability", "risk_level", "monthly_revenue", "churn_prediction"]

print(f"High/Critical Risk Customers: {len(high_risk):,}")
print(f"Combined MRR at Risk: £{high_risk[high_risk['churn_prediction']==1]['monthly_revenue'].sum():,.2f}")
print()

high_risk[display_cols].head(20)

---
## 6. Running the Dashboard

### In Colab (Preview Only)

Streamlit apps can't run interactively inside Colab notebooks. The cells above recreate the key visualisations. To run the full interactive dashboard, use your local machine.

### Locally (Full Interactive Experience)

```bash
# From the project root
pip install -r dashboard/requirements.txt
streamlit run dashboard/app.py
```

This opens the dashboard in your browser at `http://localhost:8501` with:
- Interactive sidebar filters (risk level, revenue range, probability range)
- Hover-enabled Plotly charts
- Customer drilldown with feature explorer
- Downloadable CSV of high-risk customers

### With Docker (Optional)

```bash
docker-compose up dashboard
# Opens at http://localhost:8501
```

---

## 7. Capturing Screenshots for GitHub

**Important:** Screenshots of the dashboard significantly improve GitHub repo quality. When you run the dashboard locally:

1. Open `http://localhost:8501`
2. Take screenshots of:
   - The full KPI row + risk distribution charts
   - The churn probability histogram
   - The revenue vs probability scatter plot
   - A customer drilldown example
   - The high-risk customer table

3. Save them to `docs/screenshots/` in your repo

4. Reference them in your README:
   ```markdown
   ## Dashboard Preview
   ![Dashboard Overview](docs/screenshots/dashboard_overview.png)
   ![Customer Drilldown](docs/screenshots/customer_drilldown.png)
   ```

This is one of the highest-impact things you can do for recruiter perception — a picture is worth a thousand words of README text.

---

---
## Summary — Phase 4 Complete

### What was built

| Component | File | Purpose |
|---|---|---|
| **Streamlit Dashboard** | `dashboard/app.py` | Interactive web app for customer success teams |
| **Dashboard Requirements** | `dashboard/requirements.txt` | Dependency management |
| **Colab Preview Notebook** | `notebooks/03_dashboard_demo.ipynb` | Inline preview of dashboard charts |

### Dashboard Features

- **5 KPI cards** — total customers, churners, churn rate, MRR at risk, critical count
- **4 interactive charts** — risk distribution, revenue at risk, probability histogram, scatter plot
- **Model performance summary** — metrics, threshold, revenue impact
- **Customer drilldown** — individual risk profile with feature explorer
- **High-risk table** — sortable, filterable, downloadable CSV
- **Sidebar filters** — risk level, revenue range, probability range

### Technologies demonstrated

- **Streamlit** — Industry-standard data science dashboard framework
- **Plotly** — Interactive, publication-quality visualisations
- **Product thinking** — Designed for non-technical stakeholders
- **Data storytelling** — Turns ML predictions into business actions

### What a hiring manager sees

- The candidate can make ML models **accessible to business teams**
- The dashboard answers **business questions**, not just technical ones
- There's a clear path from **prediction → action** (download high-risk list)
- The design is **professional** — not a thrown-together notebook plot

### Next: Phase 5 — Docker Containerisation & CI/CD